In [79]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [80]:
df = pd.read_csv("D:\\NLP\\Project\\Loan_Advisor_Chatbot\\Data\\raw\\loan_intents.csv")
df.head()

,text,intent
0,I want to know can someone with my income get ...,loan_eligibility
1,Why was my loan rejected right now,loan_rejection
2,Could you tell me can I close my loan early pl...,foreclosure
3,Could you help me understand how can I apply f...,loan_application
4,Could you tell me i need a loan for college in...,education_loan


In [81]:
df.shape

(3000, 2)

In [82]:
df.isnull().sum()

text      0
intent    0
dtype: int64

In [83]:
df.duplicated().sum()

np.int64(0)

# **Text Cleaning**

# **Lowercasing**

In [84]:
df['text'] = df['text'].str.lower()
df.head()

,text,intent
0,i want to know can someone with my income get ...,loan_eligibility
1,why was my loan rejected right now,loan_rejection
2,could you tell me can i close my loan early pl...,foreclosure
3,could you help me understand how can i apply f...,loan_application
4,could you tell me i need a loan for college in...,education_loan


# **Remove HTML Tags**

In [85]:
import re

def remove_html_tags(text):
    clean_text = re.sub(re.compile('<.*?>'),'',text)
    return clean_text


In [86]:
df['text'] = df['text'].apply(remove_html_tags)

# **Contraction Handling**

In [87]:
import contractions

def expand_contractions(text):
    expanded_text = contractions.fix(text)
    return expanded_text

In [88]:
df['text'] = df['text'].apply(expand_contractions)

# **Remove Punctuation**

In [89]:
pattern = r"[^a-zA-Z0-9₹%.,\s]"

def remove_punctuation(text):
    clean_text = re.sub(pattern, '', text)
    return clean_text

In [90]:
df['text'] = df['text'].apply(remove_punctuation)

# **Tokenization**

In [91]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

def tokenize_text(text):
    tokens = word_tokenize(text)
    return tokens

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Sourabh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Sourabh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [92]:
df['text'] = df['text'].apply(tokenize_text)

In [93]:
df['text']

0       [i, want, to, know, can, someone, with, my, in...
1              [why, was, my, loan, rejected, right, now]
2       [could, you, tell, me, can, i, close, my, loan...
3       [could, you, help, me, understand, how, can, i...
4       [could, you, tell, me, i, need, a, loan, for, ...
                              ...                        
2995    [please, tell, me, how, does, existing, debt, ...
2996    [please, tell, me, would, i, qualify, for, a, ...
2997    [please, what, credit, score, is, required, fo...
2998    [i, want, to, know, how, does, a, personal, lo...
2999    [could, you, tell, me, how, is, the, loan, int...
Name: text, Length: 3000, dtype: object

# **Remove Stopwords**

"Stopword removal was evaluated but not applied in the final intent-classification pipeline because function words and question words contributed to distinguishing semantically similar banking intents."

In [94]:
# from nltk.corpus import stopwords

# stop_words = set(stopwords.words('english'))
# stop_words -= {"no", "not", "nor"}

# def remove_stopwords(tokens):
#     new_text=[]
    
#     for word in tokens:
#         if word.lower() not in stop_words:
#             new_text.append(word)
#     return new_text

In [95]:
#df['text'] = df['text'].apply(remove_stopwords)

# **POS Tagging**

In [96]:
import nltk
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Sourabh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Sourabh\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Sourabh\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Sourabh\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [97]:
def pos_tagging(tokens):
    pos = pos_tag(tokens)
    return pos

In [98]:
df['pos_tags'] = df['text'].apply(pos_tagging)

In [99]:
df.head()

,text,intent,pos_tags
0,"[i, want, to, know, can, someone, with, my, in...",loan_eligibility,"[(i, NN), (want, VBP), (to, TO), (know, VB), (..."
1,"[why, was, my, loan, rejected, right, now]",loan_rejection,"[(why, WRB), (was, VBD), (my, PRP$), (loan, NN..."
2,"[could, you, tell, me, can, i, close, my, loan...",foreclosure,"[(could, MD), (you, PRP), (tell, VB), (me, PRP..."
3,"[could, you, help, me, understand, how, can, i...",loan_application,"[(could, MD), (you, PRP), (help, VB), (me, PRP..."
4,"[could, you, tell, me, i, need, a, loan, for, ...",education_loan,"[(could, MD), (you, PRP), (tell, VB), (me, PRP..."


# **Convert NLTK POS tags to WordNet POS tags**

In [100]:
def get_wordnet_pos(tag):
    
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

# **Lemmatization**

In [101]:
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(pos_tokens):
    lemmatized_words=[]
    
    for word,tag in pos_tokens:
        wordnet_pos = get_wordnet_pos(tag)
        lemma = lemmatizer.lemmatize(word,wordnet_pos)
        lemmatized_words.append(lemma)
        
    return lemmatized_words

In [102]:
df['lemmatized_token'] = df['pos_tags'].apply(lemmatize_tokens)

In [103]:
df[['text','pos_tags','lemmatized_token']].head()

,text,pos_tags,lemmatized_token
0,"[i, want, to, know, can, someone, with, my, in...","[(i, NN), (want, VBP), (to, TO), (know, VB), (...","[i, want, to, know, can, someone, with, my, in..."
1,"[why, was, my, loan, rejected, right, now]","[(why, WRB), (was, VBD), (my, PRP$), (loan, NN...","[why, be, my, loan, reject, right, now]"
2,"[could, you, tell, me, can, i, close, my, loan...","[(could, MD), (you, PRP), (tell, VB), (me, PRP...","[could, you, tell, me, can, i, close, my, loan..."
3,"[could, you, help, me, understand, how, can, i...","[(could, MD), (you, PRP), (help, VB), (me, PRP...","[could, you, help, me, understand, how, can, i..."
4,"[could, you, tell, me, i, need, a, loan, for, ...","[(could, MD), (you, PRP), (tell, VB), (me, PRP...","[could, you, tell, me, i, need, a, loan, for, ..."


In [104]:
df['clean_text'] = df['lemmatized_token'].apply(lambda X: ' '.join(X))

In [105]:
df[['text','clean_text','intent']].head()

,text,clean_text,intent
0,"[i, want, to, know, can, someone, with, my, in...",i want to know can someone with my income get ...,loan_eligibility
1,"[why, was, my, loan, rejected, right, now]",why be my loan reject right now,loan_rejection
2,"[could, you, tell, me, can, i, close, my, loan...",could you tell me can i close my loan early pl...,foreclosure
3,"[could, you, help, me, understand, how, can, i...",could you help me understand how can i apply f...,loan_application
4,"[could, you, tell, me, i, need, a, loan, for, ...",could you tell me i need a loan for college in...,education_loan


In [106]:
for i in range(10):
    print("Original :", df["text"].iloc[i])
    print("Cleaned  :", df["clean_text"].iloc[i])
    print("-" * 50)

Original : ['i', 'want', 'to', 'know', 'can', 'someone', 'with', 'my', 'income', 'get', 'a', 'loan', 'in', 'simple', 'terms']
Cleaned  : i want to know can someone with my income get a loan in simple term
--------------------------------------------------
Original : ['why', 'was', 'my', 'loan', 'rejected', 'right', 'now']
Cleaned  : why be my loan reject right now
--------------------------------------------------
Original : ['could', 'you', 'tell', 'me', 'can', 'i', 'close', 'my', 'loan', 'early', 'please']
Cleaned  : could you tell me can i close my loan early please
--------------------------------------------------
Original : ['could', 'you', 'help', 'me', 'understand', 'how', 'can', 'i', 'apply', 'for', 'a', 'loan', 'before', 'taking', 'the', 'loan']
Cleaned  : could you help me understand how can i apply for a loan before take the loan
--------------------------------------------------
Original : ['could', 'you', 'tell', 'me', 'i', 'need', 'a', 'loan', 'for', 'college', 'in', 'de

In [107]:
df['clean_text'].isnull().sum()

np.int64(0)

In [108]:
(df['clean_text'].str.len() == 0).sum()

np.int64(0)

In [109]:
df[['clean_text','intent']].head()

,clean_text,intent
0,i want to know can someone with my income get ...,loan_eligibility
1,why be my loan reject right now,loan_rejection
2,could you tell me can i close my loan early pl...,foreclosure
3,could you help me understand how can i apply f...,loan_application
4,could you tell me i need a loan for college in...,education_loan


In [110]:
df[['clean_text','intent']].isnull().sum()

clean_text    0
intent        0
dtype: int64

In [111]:
df['clean_text'].duplicated().sum()

np.int64(0)

In [112]:
df[["clean_text", "intent"]].shape

(3000, 2)

In [113]:
df[df["clean_text"].duplicated(keep=False)].sort_values("clean_text")

,text,intent,pos_tags,lemmatized_token,clean_text


In [114]:
duplicate_check = (
    df.groupby("clean_text")["intent"]
      .nunique()
      .sort_values(ascending=False)
)

duplicate_check.head(20)

clean_text
be a top up available on my exist personal loan please                             1
i would like to know what be the monthly payment for my loan before i apply        1
i would like to know what be the maximum loan amount right now                     1
i would like to know what be the maximum loan tenure before take the loan          1
i would like to know what be the maximum loan tenure for me                        1
i would like to know what be the maximum loan tenure right now                     1
i would like to know what be the minimum loan amount in detail                     1
i would like to know what be the minimum loan amount right now                     1
i would like to know what be the minimum salary require before take the loan       1
i would like to know what be the minimum salary require in detail                  1
i would like to know what be the monthly payment for my loan                       1
i would like to know what be the monthly payment for m

In [115]:
conflicts = (
    df.groupby("clean_text")["intent"]
      .nunique()
      .reset_index(name="intent_count")
)

conflicts = conflicts[conflicts["intent_count"] > 1]

conflicts

,clean_text,intent_count


In [116]:
conflict_texts = conflicts["clean_text"].tolist()

df[df["clean_text"].isin(conflict_texts)][
    ["clean_text", "intent"]
].sort_values("clean_text")

,clean_text,intent


In [118]:
df['intent'].value_counts()

intent
loan_eligibility      120
credit_score          120
greeting              120
application_status    120
loan_amount           120
car_loan              120
goodbye               120
loan_tenure           120
required_documents    120
processing_fee        120
loan_top_up           120
home_loan             120
interest_rate         120
loan_rejection        120
income_requirement    120
coapplicant           120
existing_loan         120
prepayment            120
business_loan         120
emi_calculation       120
loan_types            120
education_loan        120
loan_application      120
foreclosure           120
personal_loan         120
Name: count, dtype: int64

# Save file the file in processed folder

In [119]:
df[['clean_text','intent']].to_csv("D:\\NLP\\Project\\Loan_Advisor_Chatbot\\Data\\processed\\loan_intents_cleaned.csv", index=False)